# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and includes mass spectrometry analysis of protein abundance, modifications, and sequences from human mast cells and extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The Croissant schema describes the data as a graph of entities, with record sets containing fields and columns. All references are made by `@id`.

In [ ]:
print("Listing record sets and their fields (referenced by @id):\n")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"Record Set '@id': {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    # List all fields and their ids in this record set
    print(f"  Field @ids:")
    for field in rs.fields:
        print(f"    {field.id}: {field.name} (type: {field.data_type})")
    print("")

## Example Records Preview
Here is a sample of records (first 2) from each record set, referencing by `@id`:

In [ ]:
for rs in record_sets:
    print(f"--- Record Set: {rs.id} ---")
    for idx, rec in enumerate(dataset.records(record_set=rs.id)):
        print(f"{json.dumps(rec, indent=2) }")
        if idx >= 1:
            break
    print()

## 3. Data Extraction
Load data from specific record sets as Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

> **Tip:** Use the exact `@id` from the overview for consistent references.

In [ ]:
# Collect record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame loaded for record set {record_set_id}. Columns: {dataframes[record_set_id].columns.tolist()}")
    else:
        dataframes[record_set_id] = pd.DataFrame()

# Choose the main record set for demonstration (here we'll use the first one)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and not dataframes[main_record_set_id].empty:
    print(f"Preview of records in: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No records available to preview.")

## 4. Exploratory Data Analysis (EDA)
Let's analyze a numeric field and perform basic processing—filtering, normalizing, and grouping.

* **Choose a numeric field:** Use the field `@id` listed earlier (update below if your field of interest is different).

In [ ]:
# Example: Replace these IDs with those from your dataset if needed
record_set_id = main_record_set_id
if record_set_id is not None and not dataframes[record_set_id].empty:
    df = dataframes[record_set_id]

    # Try to identify a numeric field
    possible_numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields in {record_set_id}: {possible_numeric_fields}")
    
    # If known, use a real @id. Otherwise default to first numeric field
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else None
    if numeric_field is not None:
        threshold = df[numeric_field].mean() # As an example, filter above mean
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows")
        display(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' (z-score):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a categorical field
        possible_group_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = None
        for c in possible_group_fields:
            if df[c].nunique() > 1 and df[c].nunique() < len(df):
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
    else:
        print("No numeric fields available for analysis.")
else:
    print("No data available for EDA in the main record set.")

## 5. Visualization
Visualize distributions or relationships between key fields. Adjust field names (`@id`s) if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and not dataframes[record_set_id].empty and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[record_set_id])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have:

* Loaded FAIR^2 mass spectrometry dataset metadata and records via its Croissant schema
* Explored record sets, their fields, and reviewed some sample data using proper `@id` references
* Loaded all tabular record sets into dataframes, previewed columns, and selected them for analysis
* Performed simple exploratory filtering, normalization, and grouping on numeric fields
* Visualized the distribution and (if available) group differences of a key quantitative field

For further analysis, refer to the full Croissant metadata and reference field and record set entities by their `@id` for programmatic work and reproducibility.

> **Note:** Actual field and record set IDs may vary if the underlying dataset is updated. Always retrieve field/record set lists programmatically as demonstrated above to ensure robustness.
